# Caregiver Survey — Simplified Response Review

Two studies, same survey, 1,956 responses.

| Study | Description | Responses |
|-------|------------|----------:|
| Study 1 | **Verified caregivers** (known real) | 177 |
| Study 2 | **Online sign-ups** (need screening) | 1,779 |

This notebook walks through **five questions**:

1. What are the 14 screening rules and how do they score?
2. What do real versus flagged responses actually look like?
3. How do the scores distribute, and what does each rule catch?
4. What happens at different refusal thresholds?
5. What goes into the final spreadsheet?

In [ ]:
from pathlib import Path
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

import bot_analysis as ba

ba = importlib.reload(ba)

%matplotlib inline
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 90)

project_dir = Path.cwd()
output_dir = project_dir / "Caregiver Outputs"
cache_dir = project_dir / "data_cache"

# Load data (uses cached copy if one was pulled today)
views = ba.build_notebook_views(output_dir, cache_dir)
report_records = views["report_records"]
screen_inputs = ba.build_screen_inputs(output_dir, cache_dir)
source_table = ba.refresh_redcap_cache(project_dir, cache_dir)
audit_table = ba.check_redcap_audit_access(project_dir)

print(f"Responses loaded: {len(screen_inputs):,}")
print(screen_inputs["Group"].value_counts().to_string())

### Data sources

Both studies are pulled from the REDCap survey system. A saved copy from today is reused after its checksum is verified.

In [ ]:
display(source_table)

---
# 1. The 14 Screening Rules

Every response is checked against **14 rules**: 9 original (R1–R9) and 5 added after the review meeting (E1–E5).

Each rule is either **Serious** (5 pts) or **Mild** (1 pt).

$$
\text{Risk Score} = (\text{Serious rules violated} \times 5) + (\text{Mild rules violated} \times 1)
$$

In [ ]:
# Build the risk table (scores every response against all 14 rules)
risk_table = ba.build_risk_table(output_dir, cache_dir, project_dir)
triaged = ba.build_review_triage(risk_table)

# Rule reference card: code, meaning, severity, points, and fire rate
display(ba.build_rule_key_table(triaged))

### Threshold verification

Every timing threshold was set from the **131 verified caregivers** who finished all four sections. The table below recomputes each threshold from the raw data and confirms it matches the report.

In [ ]:
threshold_check = ba.build_threshold_check_table(output_dir, cache_dir)
display(threshold_check)
print(
    f"Recomputed from the {threshold_check.attrs['verified_n']} verified caregivers "
    f"who finished all four sections."
)

---
# 2. Real Response Comparisons

Charts are easy to argue with. Two real responses side by side are not.

- **Verified caregiver**: the one closest to the median on survey time.
- **Online sign-up**: the clearest case of each pattern.

### 2.1 Repeated-answer check (R4)

In [ ]:
# Side-by-side answers: a flagged response vs. a real caregiver
answer_comparison = ba.build_answer_comparison(screen_inputs, cache_dir)
display(answer_comparison)
ba.plot_answer_comparison(answer_comparison);

#### How the math works

The check measures the *spread* (standard deviation) of answers within a rating block. If every answer is the same number, the spread is 0. Real caregivers vary their answers, so their spread is higher.

In [ ]:
# Walk through the arithmetic for both responses
flagged_id = answer_comparison.attrs["flagged_id"]
verified_id = answer_comparison.attrs["verified_id"]
limit = answer_comparison.attrs["limit"]

for who, column in [
    (f"Online sign-up {flagged_id}", f"Online sign-up {flagged_id} answered"),
    (f"Verified caregiver {verified_id}", f"Verified caregiver {verified_id} answered"),
]:
    answers = answer_comparison[column].astype(float)
    n = len(answers)
    average = answers.mean()
    squared_gaps = ((answers - average) ** 2).sum()
    spread = (squared_gaps / (n - 1)) ** 0.5
    verdict = (
        "at or below the line → counts as repeated answers"
        if spread <= limit
        else "above the line → does not count"
    )
    print(who)
    print(f"  {n} answers         : {[int(v) for v in answers]}")
    print(f"  Average            : {average:.4f}")
    print(f"  Sum of squared gaps: {squared_gaps:.4f}")
    print(f"  Spread             : sqrt({squared_gaps:.4f} / {n - 1}) = {spread:.4f}")
    print(f"  Against the line {limit}: {verdict}")
    print()

### 2.2 Timing check — section by section

In [ ]:
# Section-by-section timing: flagged sign-up vs. real caregiver
section_comparison = ba.build_section_time_comparison(
    screen_inputs, output_dir, cache_dir
)
display(section_comparison)
ba.plot_section_time_comparison(section_comparison);

### 2.3 Family contradiction example (R8)

In [ ]:
display(ba.build_family_contradiction_example(output_dir, cache_dir))

---
# 3. Scoring Results

## 3.1 Rule Scorecard — Grid View

One row per response. Each rule is **Yes** (violated) or **No** (clean). The **Risk Score** sums points using the Mild=1, Serious=5 scale.

This matches the format:

| Record ID | R1 | R2 | … | R9 | Total Rules Violated | Risk Score (Mild=1, Serious=5) |
|-----------|----|----|---|----|--------------------|-------------------------------|

In [ ]:
rules_grid = ba.build_rules_grid(triaged)
print(f"Rows: {len(rules_grid):,}   Columns: {rules_grid.shape[1]}")
display(rules_grid.head(10))

## 3.2 Rule Scorecard — List View

Same data, but rules collapsed into one cell (easier to paste into an email).

| Record ID | Rules Violated | Total Rules Violated | Risk Score (Mild=1, Serious=5) |
|-----------|---------------|---------------------|-------------------------------|

In [ ]:
rules_list = ba.build_rules_list(triaged)
display(rules_list.head(10))

## 3.3 Summary — Actions by Group

How many responses fall into each action (Pay / Check by hand / Do not pay), split by study.

In [ ]:
display(ba.build_action_summary_table(risk_table))

## 3.4 Score Distribution

In [ ]:
display(ba.build_score_distribution_table(risk_table))

## 3.5 How Often Each Rule Fires

In [ ]:
display(ba.build_check_frequency_table(risk_table))

## 3.6 False Alarm Rate on Verified Caregivers

Any verified caregiver refused by the scoring is a mistake we know about. This table shows who was wrongly refused and why.

In [ ]:
display(ba.build_false_alarm_summary(risk_table))
display(ba.build_wrongly_refused_detail(risk_table))

## 3.7 What Happens If the Refusal Line Moves

Each row is a different threshold. The last two columns show the cost in verified caregivers wrongly refused.

In [ ]:
display(ba.build_heavy_cutoff_table(triaged))

## 3.8 Middle-Pile Triage Summary

The scores leave **~1,000 responses on 1–2 points** — too many to read individually. The triage sorts them into groups and identifies which ones a person actually needs to open.

In [ ]:
triage_summary = ba.build_triage_summary(triaged)
display(triage_summary.drop(columns=["Why"]))
print(
    f"Held responses: {triage_summary.attrs['held_total']:,}. "
    f"Responses a person actually opens: {triage_summary.attrs['total_read']}."
)

---
# 4. Final Spreadsheet

Two files come out of this notebook:

- **Master file**: all 1,956 responses with every check, scores, and actions.
- **Summary file**: the shorter overview without email addresses.

## 4.1 Preview

In [ ]:
master_export = ba.build_master_export(risk_table)
print(f"Rows: {len(master_export):,}   Columns: {master_export.shape[1]}")
display(ba.build_master_preview(master_export, rows=8))

## 4.2 Column Guide

In [ ]:
display(ba.build_master_column_guide(master_export))

## 4.3 Export

In [ ]:
master_workbook = ba.export_master_workbook(
    output_dir=output_dir,
    cache_dir=cache_dir,
    project_dir=project_dir,
    source_table=source_table,
    audit_table=audit_table,
)
print(f"Master file: {master_workbook.relative_to(project_dir)}")
print(pd.ExcelFile(master_workbook).sheet_names)

report_workbook = ba.export_simple_excel(
    output_dir=output_dir,
    cache_dir=cache_dir,
    excel_filename="ESD_Bot_Analysis_Simple_Summary.xlsx",
)
print(f"\nSummary file: {report_workbook.relative_to(project_dir)}")
print(pd.ExcelFile(report_workbook).sheet_names)

---
# 5. Key Takeaways

| Finding | Detail |
|---------|--------|
| **Time limits are defensible** | Every limit was recomputed from raw data and matched. |
| **Points separate the two studies cleanly** | 151 of 177 verified caregivers score zero. |
| **The refusal line is a choice** | Moving from 3 pts to 5 pts removes every wrongly-refused caregiver and still refuses 424 online sign-ups. |
| **The middle pile is manageable** | 1,032 held responses → 61 that someone actually reads. |
| **Arrival timing is the strongest study-level signal** | 94.2% of online sign-ups arrived within a minute of another. |
| **Email checks cost nothing** | No response used a throwaway service; 7 carry a university address. |
| **Counting Serious=5 is a real option** | At a line of 7, refuses 480 online sign-ups and zero verified caregivers. |

### Three decisions needed before payment:

1. Does an unfinished survey earn a gift card?
2. Which scoring scale: Serious=2 or Serious=5?
3. Where does the refusal line sit on that scale?